In [41]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQL Lab").getOrCreate()

In [42]:

employees_data = [
    (1, "Rahul", "IT", 70000, "Hyderabad"),
    (2, "Sneha", "HR", 60000, "Bangalore"),
    (3, "Arjun", "IT", 75000, "Chennai"),
    (4, "Priya", "Finance", 80000, "Hyderabad"),
    (5, "Karan", "IT", 50000, "Mumbai"),
    (6, "Amit", "HR", 58000, "Delhi"),
    (7, "Meera", "Finance", 82000, "Bangalore")
]

employees_cols = ["emp_id", "name", "department", "salary", "city"]

employees_df = spark.createDataFrame(employees_data, employees_cols)
employees_df.show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
+------+-----+----------+------+---------+



In [43]:
departments_data = [
    ("IT", "Technology"),
    ("HR", "People Operations"),
    ("Finance", "Accounts and Finance")
]

departments_cols = ["department", "dept_full_name"]

departments_df = spark.createDataFrame(departments_data, departments_cols)
departments_df.show()

+----------+--------------------+
|department|      dept_full_name|
+----------+--------------------+
|        IT|          Technology|
|        HR|   People Operations|
|   Finance|Accounts and Finance|
+----------+--------------------+



In [81]:
sales_data = [
    (101, 1, "Laptop", 1, 75000),
    (102, 2, "Mouse", 3, 500),
    (103, 3, "Keyboard", 2, 1500),
    (104, 1, "Monitor", 1, 12000),
    (105, 4, "Laptop", 1, 75000),
    (106, 3, "Mouse", 2, 500),
    (107, 5, "Keyboard", 1, 1500),
    (108, 1, "Laptop", 1, 75000)
]

sales_cols = ["sale_id", "emp_id", "product", "quantity", "price"]

sales_df = spark.createDataFrame(sales_data, sales_cols)
sales_df.show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|    101|     1|  Laptop|       1|75000|
|    102|     2|   Mouse|       3|  500|
|    103|     3|Keyboard|       2| 1500|
|    104|     1| Monitor|       1|12000|
|    105|     4|  Laptop|       1|75000|
|    106|     3|   Mouse|       2|  500|
|    107|     5|Keyboard|       1| 1500|
|    108|     1|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



Temperory Views are created within the same sessions

In [45]:
employees_df.createOrReplaceTempView("employees")
departments_df.createOrReplaceTempView("departments")
sales_df.createOrReplaceTempView("sales")

In [46]:
spark.sql("SELECT * FROM employees").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
+------+-----+----------+------+---------+



Global Temporary Views are used to create views across sessions

In [47]:
employees_df.createOrReplaceGlobalTempView("global_employees")

In [48]:
spark.sql("SELECT * FROM global_temp.global_employees").show()


+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
+------+-----+----------+------+---------+



In [49]:
spark.sql("CREATE DATABASE IF NOT EXISTS company_db")


DataFrame[]

In [50]:
spark.sql("SHOW DATABASES").show()

+----------+
| namespace|
+----------+
|company_db|
|   default|
+----------+



In [51]:
spark.sql("USE company_db")

DataFrame[]

In [52]:
spark.sql("SELECT current_database()").show()

+----------------+
|current_schema()|
+----------------+
|      company_db|
+----------------+



In [53]:
spark.sql("""
CREATE TABLE IF NOT EXISTS employee_master (
    emp_id INT,
    name STRING,
    department STRING,
    salary INT,
    city STRING
)
USING PARQUET
""")

spark.sql("""
INSERT INTO employee_master VALUES
(1, 'Rahul', 'IT', 70000, 'Hyderabad'),
(2, 'Sneha', 'HR', 60000, 'Bangalore'),
(3, 'Arjun', 'IT', 75000, 'Chennai'),
(4, 'Priya', 'Finance', 80000, 'Hyderabad')
""")


DataFrame[]

In [54]:
spark.sql("SELECT * FROM Employee_master").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     3|Arjun|        IT| 75000|  Chennai|
|     4|Priya|   Finance| 80000|Hyderabad|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
+------+-----+----------+------+---------+



Creating Table from a DataFrame

In [55]:
employees_df.write.mode("overwrite").saveAsTable("employee_details")

In [56]:
spark.sql("SELECT * FROM employee_details").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     4|Priya|   Finance| 80000|Hyderabad|
|     5|Karan|        IT| 50000|   Mumbai|
|     6| Amit|        HR| 58000|    Delhi|
|     7|Meera|   Finance| 82000|Bangalore|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
+------+-----+----------+------+---------+



In [57]:
spark.sql("SHOW tables in company_db").show()

+----------+----------------+-----------+
| namespace|       tableName|isTemporary|
+----------+----------------+-----------+
|company_db|employee_details|      false|
|company_db| employee_master|      false|
|company_db|high_paid_worker|      false|
|          |     departments|       true|
|          |       employees|       true|
|          |           sales|       true|
+----------+----------------+-----------+



In [58]:
spark.sql("""CREATE OR REPLACE VIEW HIGH_PAID_WORKER
 AS SELECT * FROM employee_details
  WHERE salary > 70000""")

DataFrame[]

In [59]:
spark.sql("SELECT * FROM HIGH_PAID_WORKER").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     4|Priya|   Finance| 80000|Hyderabad|
|     7|Meera|   Finance| 82000|Bangalore|
|     3|Arjun|        IT| 75000|  Chennai|
+------+-----+----------+------+---------+



In [60]:
spark.sql(" SELECT * from HIGH_PAID_WORKER where city in('Chennai', 'Hyderabad')").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     4|Priya|   Finance| 80000|Hyderabad|
|     3|Arjun|        IT| 75000|  Chennai|
+------+-----+----------+------+---------+



In [61]:
spark.sql("""Select * from employee_details where department = 'IT' and city = 'Hyderabad'""").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     1|Rahul|        IT| 70000|Hyderabad|
+------+-----+----------+------+---------+



In [77]:
employees_df.createOrReplaceTempView("employees_df")
departments_df.createOrReplaceTempView("departments_df")
sales_df.withColumnRenamed("emp_id", "emp_id_sales")

DataFrame[sale_id: bigint, emp_id_sales: bigint, product: string, quantity: bigint, price: bigint]

In [68]:
spark.sql("""
SELECT
    e.emp_id, e.name, e.department, d.dept_full_name
    from employees_df e left join departments_df d
    on e.department = d.department
""").show()

+------+-----+----------+--------------------+
|emp_id| name|department|      dept_full_name|
+------+-----+----------+--------------------+
|     2|Sneha|        HR|   People Operations|
|     1|Rahul|        IT|          Technology|
|     3|Arjun|        IT|          Technology|
|     6| Amit|        HR|   People Operations|
|     4|Priya|   Finance|Accounts and Finance|
|     7|Meera|   Finance|Accounts and Finance|
|     5|Karan|        IT|          Technology|
+------+-----+----------+--------------------+



In [69]:
spark.sql("""
SELECT
    Count(*) as total_employees,
    SUM(salary) as total_salary,
    AVG(salary) as avg_salary,
    MAX(salary) as max_salary,
    MIN(salary) as min_salary
    FROM employees_df"""
).show()


+---------------+------------+-----------------+----------+----------+
|total_employees|total_salary|       avg_salary|max_salary|min_salary|
+---------------+------------+-----------------+----------+----------+
|              7|      475000|67857.14285714286|     82000|     50000|
+---------------+------------+-----------------+----------+----------+



In [74]:
spark.sql("""
SELECT
department, count(*)  as total_employees
FROM employees_df
GROUP BY department
""").show()

+----------+---------------+
|department|total_employees|
+----------+---------------+
|        HR|              2|
|        IT|              3|
|   Finance|              2|
+----------+---------------+



In [96]:
spark.sql("""
SELECT department, avg(salary) as avg_salary
FROM employees_df
GROUP BY department
""").show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|        HR|   59000.0|
|        IT|   65000.0|
|   Finance|   81000.0|
+----------+----------+



In [97]:
spark.sql("select * from sales ").show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|    101|     1|  Laptop|       1|75000|
|    102|     2|   Mouse|       3|  500|
|    103|     3|Keyboard|       2| 1500|
|    104|     1| Monitor|       1|12000|
|    105|     4|  Laptop|       1|75000|
|    106|     3|   Mouse|       2|  500|
|    107|     5|Keyboard|       1| 1500|
|    108|     1|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



In [95]:
spark.sql("""
select e.name, sum(s.quantity * s.price) as total_sales
from employees_df e inner join sales s
on e.emp_id = s.emp_id
group by e.name
""").show()

+-----+-----------+
| name|total_sales|
+-----+-----------+
|Sneha|       1500|
|Priya|      75000|
|Rahul|     162000|
|Arjun|       4000|
|Karan|       1500|
+-----+-----------+



HAVING

In [85]:
spark.sql("""
    select department, avg(salary) as avg_salary
    from employees_df
    group by department
    having avg(salary) > 70000
    """
).show()

+----------+----------+
|department|avg_salary|
+----------+----------+
|   Finance|   81000.0|
+----------+----------+



In [94]:
spark.sql("""
select product, sum(quantity) as total_qty
from sales
group by product
having sum(quantity) > 2
""").show()

+--------+---------+
| product|total_qty|
+--------+---------+
|  Laptop|        3|
|   Mouse|        5|
|Keyboard|        3|
+--------+---------+



ORDER BY

In [91]:
spark.sql("""
select *
from employees
order by salary desc""").show()

+------+-----+----------+------+---------+
|emp_id| name|department|salary|     city|
+------+-----+----------+------+---------+
|     7|Meera|   Finance| 82000|Bangalore|
|     4|Priya|   Finance| 80000|Hyderabad|
|     3|Arjun|        IT| 75000|  Chennai|
|     1|Rahul|        IT| 70000|Hyderabad|
|     2|Sneha|        HR| 60000|Bangalore|
|     6| Amit|        HR| 58000|    Delhi|
|     5|Karan|        IT| 50000|   Mumbai|
+------+-----+----------+------+---------+



In [93]:
spark.sql("""
Select product, sum(price * quantity) as total_price
from sales
group by product
order by total_price desc
""").show()

+--------+-----------+
| product|total_price|
+--------+-----------+
|  Laptop|     225000|
| Monitor|      12000|
|Keyboard|       4500|
|   Mouse|       2500|
+--------+-----------+



WINDOW FUNCTION

RANK() provides same rank for equal values and the next one is given a skipped rank

EG: 1,2,3,3,5

In [98]:
spark.sql("""
select emp_id, name, salary, department,
RANK() OVER (PARTITION BY department ORDER BY salary desc) as rank
from employees
""").show()

+------+-----+------+----------+----+
|emp_id| name|salary|department|rank|
+------+-----+------+----------+----+
|     7|Meera| 82000|   Finance|   1|
|     4|Priya| 80000|   Finance|   2|
|     2|Sneha| 60000|        HR|   1|
|     6| Amit| 58000|        HR|   2|
|     3|Arjun| 75000|        IT|   1|
|     1|Rahul| 70000|        IT|   2|
|     5|Karan| 50000|        IT|   3|
+------+-----+------+----------+----+



DENSE_RANK() provides same rank for equal values and the next one is given continued rank

EG: 1,2,3,3,3,4,5

In [104]:
spark.sql(
    """
    select emp_id, name, salary, department,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary desc) as dense_rank
    from employees
    """
).show()

+------+-----+------+----------+----------+
|emp_id| name|salary|department|dense_rank|
+------+-----+------+----------+----------+
|     7|Meera| 82000|   Finance|         1|
|     4|Priya| 80000|   Finance|         2|
|     2|Sneha| 60000|        HR|         1|
|     6| Amit| 58000|        HR|         2|
|     3|Arjun| 75000|        IT|         1|
|     1|Rahul| 70000|        IT|         2|
|     5|Karan| 50000|        IT|         3|
+------+-----+------+----------+----------+



ROW_NUMBER() provides contiguous numbers for rows w.r.t order by function.
It gives seperate numbers for identitical values

EG: 1,2,3,4,5

In [103]:
spark.sql(
    """
    select
    emp_id, name, salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary desc) as row_num
    from employees
    """
).show()

+------+-----+------+-------+
|emp_id| name|salary|row_num|
+------+-----+------+-------+
|     7|Meera| 82000|      1|
|     4|Priya| 80000|      2|
|     2|Sneha| 60000|      1|
|     6| Amit| 58000|      2|
|     3|Arjun| 75000|      1|
|     1|Rahul| 70000|      2|
|     5|Karan| 50000|      3|
+------+-----+------+-------+

